In [273]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd
from scipy.interpolate import griddata

In [274]:
# df = pd.read_csv(r'C:\\Users\\canel\\OneDrive\Desktop\\ToyData.csv', skiprows=3)
df = pd.read_csv('ToyData2.csv')
print(df.shape)
df.head(20)

(2400, 6)


,id,instance_no,reg_type,k_repetition,reg_value,loss
0,0,0.0,L2,0.0,0.0010,1.087530
1,1,0.0,L2,1.0,0.0010,0.956053
2,2,0.0,L2,2.0,0.0010,0.984344
3,3,0.0,L2,3.0,0.0010,1.007919
4,4,0.0,L2,4.0,0.0010,0.965957
5,5,0.0,L2,5.0,0.0010,0.999569
6,6,0.0,L2,6.0,0.0010,1.095158
7,7,0.0,L2,7.0,0.0010,0.945276
8,8,0.0,L2,8.0,0.0010,0.924677
9,9,0.0,L2,9.0,0.0010,0.907467


In [275]:
unique_b_no = df['instance_no'].unique()
unique_reg_types = df['reg_type'].unique()
unique_reg_values = df.groupby('reg_type')['reg_value'].unique()
# unique_reg1, unique_reg2 = unique_reg_values[0], unique_reg_values[1]
print(unique_b_no)
print(unique_reg_types)
print(unique_reg_values)
print(df.columns)

[ 0.  1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12. 13. 14. 15. 16. 17.
 18. 19.]
['L2' 'dropout']
reg_type
L2         [0.001, 0.0505, 0.1]
dropout      [0.05, 0.15, 0.25]
Name: reg_value, dtype: object
Index(['id', 'instance_no', 'reg_type', 'k_repetition', 'reg_value', 'loss'], dtype='object')


In [276]:
summary_df = pd.DataFrame(columns=['instance_no','reg_type', 'reg_value', 'mean_loss', 'stdv_loss'])

for b_value in unique_b_no:
    # print(b_value)

    for reg_type in unique_reg_types:
        # print(reg_type)

        for reg_values in unique_reg_values:
            # print(reg_values)

            for value in reg_values:
                # print(value)
                selected_rows = df[(df['instance_no'] == b_value) & (df['reg_type'] == reg_type) & (df['reg_value'] == value)]
                # print(selected_rows.shape)
                # input()
                mean_loss = selected_rows['loss'].mean()
                stdv_loss = selected_rows['loss'].std()
                # print(f'{b_value}, {reg_type}, {reg_value}, {mean_loss}')
                new_row = {'instance_no': b_value, 'reg_type': reg_type, 'reg_value': value, 'mean_loss': mean_loss, 'stdv_loss': stdv_loss}
                summary_df = pd.concat([summary_df, pd.DataFrame([new_row])], ignore_index=True)


summary_df = summary_df.dropna()
print(f'Summary shape: {summary_df.shape}')
print(summary_df.head(60))
# print(summary_df.tail(10))

Summary shape: (120, 5)
     instance_no reg_type  reg_value  mean_loss  stdv_loss
0            0.0       L2     0.0010   1.011106   0.091839
1            0.0       L2     0.0505   1.010189   0.086318
2            0.0       L2     0.1000   1.100653   0.122635
9            0.0  dropout     0.0500   1.039677   0.116903
10           0.0  dropout     0.1500   1.218006   0.080201
11           0.0  dropout     0.2500   1.699314   0.093512
12           1.0       L2     0.0010   0.389934   0.081680
13           1.0       L2     0.0505   0.395184   0.111245
14           1.0       L2     0.1000   0.472614   0.103909
21           1.0  dropout     0.0500   0.333108   0.129785
22           1.0  dropout     0.1500   0.606454   0.104014
23           1.0  dropout     0.2500   0.973427   0.115903
24           2.0       L2     0.0010   0.135266   0.089786
25           2.0       L2     0.0505   0.159965   0.097808
26           2.0       L2     0.1000   0.198298   0.070145
33           2.0  dropout     0.

In [277]:
summary_df.columns

Index(['instance_no', 'reg_type', 'reg_value', 'mean_loss', 'stdv_loss'], dtype='object')

In [278]:
# summary_unique_reg_types = df['reg_type'].unique()
# # print(summary_unique_reg_types)

# def split_dataframe(df, feature, value):
#     df_split = df([df[feature] == value])
#     return df_split    

In [279]:
# Assuming you have a dataframe called summary_df
# with columns ['instance_no', 'reg_type', 'reg_value', 'mean_loss', 'stdv_loss']

# Create a dictionary of dataframes to store the split dataframes
split_dataframes = {}

# Group the dataframe by 'reg_type'
grouped = summary_df.groupby('reg_type')

# Iterate over each group
for reg_type, group_df in grouped:
    # Store the dataframe with the current 'reg_type' in the dictionary
    split_dataframes[reg_type] = group_df

# Now split_dataframes contains multiple dataframes, each with a unique value of 'reg_type'



In [280]:

# len(split_dataframes)

In [281]:
# split_dataframes['L2'].shape

In [282]:
def plot_3d(df_dict, output_variable):
    
    for key, df in df_dict.items():
        # print(f'Key {key}')
        # print(f'Value: {df}') 

        # Define grid for surface
        b_no_range = np.linspace(df['instance_no'].min(), df['instance_no'].max(), 100)
        reg_value_range = np.linspace(df['reg_value'].min(), df['reg_value'].max(), 100)
        b_no_grid, reg_value_grid = np.meshgrid(b_no_range, reg_value_range)

        # Interpolate mean_loss values for the surface
        
        # print(f'Len instance: {df['instance_no'].shape}')
        # print(f'Len reg val: {df['reg_value'].shape}')
        
        mean_loss_surface = griddata((df['instance_no'], df['reg_value']), 
                                    df[output_variable], 
                                    (b_no_grid, reg_value_grid), 
                                    method='cubic')

        # Create the 3D surface plot
        fig = go.Figure()

        # Add scatter data points
        fig.add_trace(go.Scatter3d(
            x=df['instance_no'],
            y=df['reg_value'],
            z=df[output_variable],
            mode='markers',
            marker=dict(
                size=3,
                color='blue',
                opacity=0.5
            ),
            name='Data Points'
        ))

        # Add surface plot
        fig.add_trace(go.Surface(
            x=b_no_range,
            y=reg_value_range,
            z=mean_loss_surface,
            colorscale='Jet',
            opacity=0.9
        ))

        # Update layout for better visibility
        fig.update_layout(scene=dict(
                            xaxis_title='instance_no',
                            yaxis_title='reg_value',
                            zaxis_title= output_variable),
                            width=800,  # adjust width
                            height=600, # adjust height
                            title=f'Mean Loss for different values of {key} regularization')

        # Show the plot
        fig.show()


In [283]:
plot_3d(split_dataframes, 'mean_loss')

In [284]:
# plot_3d(split_dataframes, 'stdv_loss')

In [285]:
# Group by 'reg_type' and 'reg_value'
grouped = summary_df.groupby('reg_type')#, 'reg_value'])

# Create a dictionary to store DataFrames for each combination
dfs = {}

# Iterate over the groups and create a DataFrame for each combination
for name, group in grouped:
    reg_type = name
    df_name = f"{reg_type}"  # Name for the DataFrame
    dfs[df_name] = group.copy()  # Copy the group DataFrame and store it


In [286]:
# for df_name, df in dfs.items():
#     print(f"DataFrame: {df_name}")
#     print(df.shape)
#     print(df)
#     print("\n")

In [287]:
# def plot_2d(df, )

In [288]:
# Group by 'reg_type'
grouped_by_reg_type = summary_df.groupby('reg_type')

# Iterate over each group of reg_type
for reg_type, reg_type_group in grouped_by_reg_type:
    # Group by 'reg_value' within each reg_type group
    grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
    # Create a figure for the current reg_type
    fig = go.Figure()
    
    # Iterate over each group of reg_value within the current reg_type group
    for reg_value, reg_value_group in grouped_by_reg_value:
        fig.add_trace(go.Scatter(x=reg_value_group['instance_no'], y=reg_value_group['mean_loss'], 
                                 mode='lines+markers', name=reg_value,
                                 marker=dict(size=7)))
    
    fig.update_layout(legend=dict(title="Reg. Value"))
    fig.update_layout(title=f'Iterations vs. mean loss for {reg_type}',
                      xaxis_title='iteration',
                      yaxis_title='mean_loss',
                      width=600,  # Set width of the plot
                      height=400)  # Set height of the plot
    
    # Show the plot for the current reg_type
    fig.show()


In [289]:
# Group by 'reg_type'
# grouped_by_reg_type = value.groupby('reg_type')

# Iterate over each group of reg_type
for reg_type, reg_type_group in grouped_by_reg_type:
    # Group by 'reg_value' within each reg_type group
    grouped_by_reg_value = reg_type_group.groupby('reg_value')
    
    # Create a figure for the current reg_type
    fig = go.Figure()
    
    # Iterate over each group of reg_value within the current reg_type group
    for reg_value, reg_value_group in grouped_by_reg_value:
        fig.add_trace(go.Scatter(x=reg_value_group['instance_no'], y=reg_value_group['stdv_loss'], 
                                 mode='lines+markers', name=reg_value,
                                 marker=dict(size=7)))
    
    fig.update_layout(legend=dict(title="Reg. Value"))
    fig.update_layout(title=f'Iterations vs. stdv loss for {reg_type}',
                      xaxis_title='iteration',
                      yaxis_title='stdv_loss',
                      width=600,  # Set width of the plot
                      height=400)  # Set height of the plot
    
    # Show the plot for the current reg_type
    fig.show()
